In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd
repo=Path('/tmp/PCC')
subprocess.run(['git','clone','--quiet','https://github.com/changxinjiresearch/PCC.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--quiet','8ba4df4'],check=True)
sys.path.insert(0,str(repo))
inputs=Path('/kaggle/input'); working=Path('/kaggle/working/pcc_internal_completion_2026'); analysis=working/'analysis'
(analysis/'03_imperfect_guidance').mkdir(parents=True,exist_ok=True)
core_candidates=list(inputs.rglob('MECHANISM_CASE_METRICS.csv')); assert core_candidates, 'core result source missing'; core_root=core_candidates[0].parents[1]
for rel in ['01_mechanism_ablation/MECHANISM_CASE_METRICS.csv','02_shuffled_target/SHUFFLED_TARGET_CASE_METRICS.csv','04_target_construction/TARGET_CONSTRUCTION_CASE_METRICS.csv']:
 src=core_root/rel; dst=analysis/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
difference_candidates=list(inputs.rglob('DIFFERENCE_MAP_GATE_STATUS.json')); assert difference_candidates, 'difference-control result source missing'; difference_root=difference_candidates[0].parent
for name in ['DIFFERENCE_MAP_GATE_STATUS.json','DIFFERENCE_MAP_CASE_METRICS.csv']:
 src=difference_root/name; dst=analysis/'05_difference_map_control'/name; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
repeat_files=list(inputs.rglob('IMPERFECT_GUIDANCE_REPEAT_METRICS.csv')); assert len(repeat_files)==2, repeat_files
repeats=pd.concat([pd.read_csv(p) for p in repeat_files],ignore_index=True).sort_values(['case_id','condition','method','repeat'],kind='stable')
assert len(repeats)==5160 and repeats.case_id.nunique()==40; repeats.to_csv(analysis/'03_imperfect_guidance/IMPERFECT_GUIDANCE_REPEAT_METRICS.csv',index=False)
frozen_candidates=[p.parent for p in inputs.rglob('LOCKED_CASE_MANIFEST.csv') if (p.parent/'ALL_CASE_METHOD_METRICS.csv').exists()]; assert frozen_candidates, 'frozen v8 root missing'; frozen=frozen_candidates[0]
subprocess.run([sys.executable,'-m','experiments.finalize_internal_completion_2026','--output-root',str(analysis),'--frozen-reference',str(frozen)],cwd=repo,check=True)
alias=working/'frozen_alias'; alias.mkdir(parents=True,exist_ok=True)
(alias/'held_out_p0').symlink_to(frozen/'held_out_p0',target_is_directory=True); (alias/'retrospective').symlink_to(frozen/'retrospective',target_is_directory=True)
manifest=pd.read_csv(frozen/'LOCKED_CASE_MANIFEST.csv')
for column in ['current_t1c_path','current_mask_path','future_mask_path']:
 manifest[column]=manifest[column].str.replace('/kaggle/input/datasets/stacyvangepuram/mu-glioma-post','/kaggle/input/mu-glioma-post',regex=False)
assert all(Path(p).exists() for column in ['current_t1c_path','current_mask_path','future_mask_path'] for p in manifest[column])
manifest.to_csv(alias/'LOCKED_CASE_MANIFEST.csv',index=False)
subprocess.run([sys.executable,'-m','experiments.generate_internal_qualitative_panels','--frozen-root',str(alias),'--analysis-root',str(analysis),'--output-root',str(working)],cwd=repo,check=True)
out=working/'07_qualitative_panels'; assert len(list(out.glob('*.png')))==6 and len(list(out.glob('*.svg')))==6
status={'status':'COMPLETE','panels':6,'selection':'prespecified deterministic case-level ranks','training_performed':False,'predictor_retrained':False,'p0_regenerated':False}
(out/'LAYER2_PANELS_COMPLETE.json').write_text(json.dumps(status,indent=2)+'\n')
